# Logistic Regression Stacking

**ISIC 2024 – Skin Cancer Detection with 3D-TBP**

This notebook trains the Logistic Regression meta-model used to combine the predictions from the three base models:

- ResNet18
- EfficientNet-B0
- CatBoost

The stacking model learns how to combine the prediction probabilities produced by each model in order to maximize the overall classification performance.

ResNet18 --------\  
EfficientNet -------> Logistic Regression ---> Final Prediction                
CatBoost --------/  

## Why Stacking?

Each base model captures different aspects of the data.

- **ResNet18** learns visual representations from dermoscopic images.
- **EfficientNet-B0** provides a complementary image representation.
- **CatBoost** exploits patient metadata and handcrafted features.

Rather than averaging their predictions, a Logistic Regression model is trained to learn the optimal combination of these three probability estimates.

## Configuration

Define the paths to the Out-of-Fold prediction files generated by each base model together with the training hyperparameters for the Logistic Regression stacker.

In [ ]:
# =========================
# Competition paths
# =========================

TEST_CSV = "/kaggle/input/competitions/isic-2024-challenge/test-metadata.csv"
TEST_HDF5 = "/kaggle/input/competitions/isic-2024-challenge/test-image.hdf5"
SAMPLE_SUBMISSION_CSV = "/kaggle/input/competitions/isic-2024-challenge/sample_submission.csv"


# =========================
# Image model
# =========================

EFF_MODEL_DIR = "/kaggle/input/datasets/wagneraugustoaff/isic2024-efficientnetb0-1fold"
RSN_MODEL_DIR = "/kaggle/input/datasets/wagneraugustoaff/isic2024-resnet18-5fold-baseline"

IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
PIN_MEMORY = True


# =========================
# CatBoost model
# =========================

CAT_MODEL_DIR = "/kaggle/input/datasets/wagneraugustoaff/isic2024-catboost-5fold-baseline/new_models/catboost"

FEATURES_PATH = f"{CAT_MODEL_DIR}/features.json"
N_FOLDS = 5

## Imports

Import the required libraries for data processing, model training, and evaluation.

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import transforms
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# CODE_PATH = "/kaggle/input/isic2024-code/src/stacking"
# sys.path.append(CODE_PATH)

# from stack_train import train_cv

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
test_df = pd.read_csv(TEST_CSV)

## Load Out-of-Fold Predictions

Load the Out-of-Fold prediction probabilities generated during cross-validation.

Using OOF predictions prevents information leakage because every prediction was produced by a model that never saw the corresponding training sample.

In [ ]:
OOF_DIR = Path(RSN_MODEL_DIR)

files = sorted(OOF_DIR.glob("oof_fold*.csv"))

oof_resnet = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True,
)

oof_resnet = oof_resnet.rename(
    columns={"prediction": "resnet"}
)

oof_resnet.to_csv(
    "oof_resnet.csv",
    index=False,
)

oof_resnet

In [ ]:
OOF_DIR = Path(EFF_MODEL_DIR)

files = sorted(OOF_DIR.glob("oof_fold*.csv"))

oof_efficientnet = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True,
)

oof_efficientnet = oof_efficientnet.rename(
    columns={"prediction": "efficientnet"}
)

oof_efficientnet.to_csv(
    "oof_efficientnet.csv",
    index=False,
)

oof_efficientnet

In [ ]:
OOF_DIR = Path(CAT_MODEL_DIR)

files = sorted(OOF_DIR.glob("oof_predictions.csv"))

oof_catboost = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True,
)

oof_catboost = oof_catboost.rename(
    columns={"prediction": "catboost"}
)

oof_catboost.to_csv(
    "oof_catboost.csv",
    index=False,
)

oof_catboost

In [ ]:
resnet = pd.read_csv("/kaggle/working/oof_resnet.csv")
effnet = pd.read_csv("/kaggle/working/oof_efficientnet.csv")
catboost = pd.read_csv("/kaggle/working/oof_catboost.csv")

## Build the Stacking Dataset

Merge the prediction probabilities from the three base models into a single feature matrix used to train the Logistic Regression meta-model.

In [ ]:
stack = (
    resnet[
        ["isic_id", "target", "resnet"]
    ]
    .merge(
        effnet[
            ["isic_id", "efficientnet"]
        ],
        on="isic_id",
    )
    .merge(
        catboost[
            ["isic_id", "catboost"]
        ],
        on="isic_id",
    )
)

In [ ]:
# Each column corresponds to one base-model probability.
X = stack[[
    "resnet",
    "efficientnet",
    "catboost"
]]

y = stack["target"]

## Cross Validation

Evaluate the stacking model using the same five-fold split adopted during base-model training.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

oof_pred = np.zeros(len(X))

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
    )

    model.fit(X_train, y_train)

    # Store validation predictions in their original positions.
    oof_pred[valid_idx] = model.predict_proba(
        X_valid
    )[:,1]

## Cross-Validation Summary

Aggregate the fold-level metrics and report the final Out-of-Fold performance of the stacking model.

In [ ]:
pauc = roc_auc_score(
    y,
    oof_pred,
    max_fpr=0.2,
)

pauc

## Final Meta-Model Training

After validating the stacking strategy through cross-validation, train a new Logistic Regression model using the complete set of Out-of-Fold predictions.

This model is the one used during inference to combine the predictions from the three base models on the competition test set.

In [ ]:
meta_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)

meta_model.fit(X, y)

import joblib

joblib.dump(meta_model, "/kaggle/working/logistic_regression.pkl")

## Learned Weights

Inspect the Logistic Regression coefficients to understand the relative contribution of each base model to the final prediction.

In [ ]:
logistic_pred = meta_model.predict_proba(X)[:, 1]

pd.DataFrame(
    {
        "Model": ["ResNet18", "EfficientNet-B0", "CatBoost"],
        "Weight": meta_model.coef_[0],
    }
)

## Model Comparison

Compare the performance of the individual models, the weighted-average ensemble, and the Logistic Regression stacker.

This comparison highlights the contribution of multimodal stacking over simpler ensemble strategies.

In [ ]:
W_RESNET = 0.20
W_EFFICIENTNET = 0.45
W_CATBOOST = 0.35

stack["weighted_pred"] = (
    W_RESNET * stack["resnet"]
    + W_EFFICIENTNET * stack["efficientnet"]
    + W_CATBOOST * stack["catboost"]
)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "ResNet18",
        "EfficientNet-B0",
        "CatBoost",
        "Weighted Average",
        "Logistic Regression",
    ],
    "pAUC": [
        roc_auc_score(stack["target"], stack["resnet"], max_fpr=0.2),
        roc_auc_score(stack["target"], stack["efficientnet"], max_fpr=0.2),
        roc_auc_score(stack["target"], stack["catboost"], max_fpr=0.2),
        roc_auc_score(stack["target"], stack["weighted_pred"], max_fpr=0.2),
        roc_auc_score(stack["target"], logistic_pred, max_fpr=0.2),
    ],
})

print(results.sort_values("pAUC", ascending=False))

In [ ]:
# X = stack[
#     [
#         "resnet",
#         "efficientnet",
#         "catboost",
#     ]
# ].values

# y = stack["target"].values

In [ ]:
# results = train_cv(

#     X=X,

#     y=y,

#     device=device,

#     epochs=30,

#     learning_rate=1e-3,

#     batch_size=1024,

#     save_dir="/kaggle/working/nn"

# )

### Performance Summary

The Logistic Regression stacker achieved the highest Out-of-Fold pAUC, demonstrating that learning the optimal combination of the three base models provides a measurable improvement over simple averaging strategies.